# Chapter 7 &mdash; Subset Construction: NFA to DFA

**Concept 8 of the Chapter 7 decomposition:** *Subset Construction: Converting an NFA to a DFA*

DFA states are <i>sets</i> of NFA states; expand each unexpanded set by Eclose&ndash;move&ndash;Eclose until closed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Subset-Construction/Concept-Subset-Construction.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The **subset construction** makes the token set *itself* the DFA state.

* start state: $Eclosure(Q_0)$;
* for each unexpanded set $S$ and symbol $a$: $Eclosure(\delta(S,a))$ is a new state;
* final: any set meeting $F$;
* repeat until no new set appears.

At most $2^{|Q|}$ sets exist, so it terminates &mdash; and that bound is where the
exponential blow-up of Chapter 5 comes from. In practice only the **reachable** sets
appear, which is usually far fewer.

This construction *is* the proof that NFA add no power.

## 2. Definitions

### The NFA to determinize

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> B
B : 0 | 1 -> F
''')

### The construction, written out

In [ ]:
def subset(N):
    start = frozenset(Eclosure(N, N["Q0"]))
    Q, todo, Dl = {start}, [start], {}
    while todo:
        Sset = todo.pop()
        for a in sorted(N["Sigma"]):
            t = frozenset(Eclosure(N, {x for q in Sset for x in step_nfa(N, q, a)}))
            Dl[(Sset, a)] = t
            if t not in Q: Q.add(t); todo.append(t)
    F = {s for s in Q if s & N["F"]}
    return Q, Dl, start, F

## 3. Tests

The reachable subsets, listed.

In [ ]:
Q, Dl, start, F = subset(N)
print("start set :", sorted(start))
for s in sorted(Q, key=lambda x: (len(x), sorted(x))):
    print("  %-22s %s" % (sorted(s), "FINAL" if s in F else ""))
print("\n%d reachable subsets out of 2^%d = %d possible"
      % (len(Q), len(N["Q"]), 2 ** len(N["Q"])))

`nfa2dfa` produces a machine of the same size and the same language.

In [ ]:
D = nfa2dfa(N)
print("nfa2dfa gives %d states; our construction found %d" % (len(D["Q"]), len(Q)))
assert len(D["Q"]) == len(Q)
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_dfa(D, s) == accepts_nfa(N, s) for s in strs)
print("DFA and NFA agree on all %d strings up to length 10" % len(strs))

Only **reachable** subsets appear &mdash; the $2^{|Q|}$ bound is worst case, not typical.

In [ ]:
print("2^|Q| = %d, reachable = %d, minimized = %d"
      % (2 ** len(N["Q"]), len(D["Q"]), len(min_dfa(D)["Q"])))

But the worst case is real: the $N$-th-last family hits it.

In [ ]:
def nth_last_nfa(k):
    lines = ['NFA', 'I : 0 | 1 -> I', 'I : 1 -> S1']
    for j in range(1, k-1): lines.append('S%d : 0 | 1 -> S%d' % (j, j+1))
    if k > 1: lines.append('S%d : 0 | 1 -> F' % (k-1))
    else:     lines = ['NFA', 'I : 0 | 1 -> I', 'I : 1 -> F']
    return md2mc('\n'.join(lines))

for k in range(1, 5):
    A = nth_last_nfa(k)
    print("k=%d : NFA %2d states -> DFA %2d states (min %2d), 2^k = %d"
          % (k, len(A["Q"]), len(nfa2dfa(A)["Q"]), len(min_dfa(nfa2dfa(A))["Q"]), 2**k))
    assert len(min_dfa(nfa2dfa(A))["Q"]) >= 2**k

Names get long; `shrink_dfastates` renames them.

In [ ]:
print("a raw subset-construction state name :", sorted(D["Q"])[0][:60], "...")

## 4. Animation

The determinized machine &mdash; each state is a whole token set.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(N)), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Determinize the $\varepsilon$-NFA of Concept 6 by hand. How many subsets are reachable?
2. Why is $\emptyset$ a legitimate subset-construction state? What is it?
3. When does the subset construction produce **fewer** states than the NFA had?

In [ ]:
# Your work for the exercises above.